<div dir="rtl">

# אימון מחדש של גלאי האבטחה (`security.pt`): סכין, אקדח, אש, עשן

**למה:** במדידה על UCF-Crime ‏(`live/results/ucf_2026-11-24`) הגלאי הנוכחי לא ראה סכין באף אחד מ-46 האירועים, וראה אקדח רק ב-20% מהם.

**ניסיון 1 נכשל:** ב-WeaponDetection היו רק 7 סכינים, והגלאי יצא גרוע מהישן. הפרטים ב-CHANGELOG. **זה ניסיון 2.**

**מה המחברת עושה, בלי שום תיוג ידני:**
1. מורידה שלושה מקורות:
   - **Dangerous Items** מ-Zenodo: כ-8,500 תמונות עם סכינים, מצ'טות, אקדחים, רובים ומחבטים. יש בו גם חפצים קטנים, מוסתרים, מטושטשים וחשוכים.
   - אקדחים ממצלמות אבטחה אמיתיות.
   - D-Fire לאש ולעשן.
2. מצ'טה נספרת כסכין. תמונה עם רובה נזרקת, כי "אקדח" אצלנו הוא אקדח קצר. מחבט נשאר בתור רקע.
3. לכל תמונת אימון עם סכין או אקדח מוסיפה עותק "מצלמת אבטחה": רזולוציה נמוכה, טשטוש ודחיסה.
4. **בדיקה לפני האימון:** אם יש פחות מ-1,500 תיבות במחלקה כלשהי, המחברת עוצרת ולא מאמנת.
5. ממשיכה מ-`security.pt` עם קצב למידה נמוך, ומקפיאה את השכבות הראשונות, כדי לא לשכוח אש ועשן.
6. בודקת על UCF את הגלאי הישן מול החדש. UCF לא נמצא באימון.

**לפני שמתחילים:** Runtime → Change runtime type → **T4 GPU**. ואז Runtime → Run all.

</div>

In [ ]:
# 1. GPU, Drive, packages
%cd /content
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ none: Runtime → Change runtime type → T4 GPU')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
RUNS = f'{DRIVE}/security_train'            # training checkpoints: survive a disconnect
os.makedirs(RUNS, exist_ok=True)
!pip -q install ultralytics lap datasets huggingface_hub pyarrow
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF token: ok')
except Exception:
    print('no HF_TOKEN secret: continuing without it')

In [ ]:
# 2. Code (the branch with the new dataset script)
BRANCH = 'live-action-alerts'
%cd /content
!rm -rf /content/final_project
!git clone -q -b {BRANCH} https://github.com/Talyaavraham/final_project.git /content/final_project
%cd /content/final_project/backend
import sys; sys.path.insert(0, 'train')
!ls security.pt train/security_dataset.py live/perceive.py

In [ ]:
# 3. Download the sources (~6 GB). The Dangerous Items zip is kept on Drive, so it downloads once.
from huggingface_hub import hf_hub_download, snapshot_download
hf_hub_download('jsalazar/US-Real-time-gun-detection-in-CCTV-An-open-problem-dataset', 'weapons_images_2fps.zip',
                repo_type='dataset', local_dir='/content/src/weapons')
snapshot_download('badsaarow/d-fire', repo_type='dataset', local_dir='/content/src/dfire', allow_patterns=['data/*.parquet'])
DZ = f'{DRIVE}/security_sources/dangerous_items.zip'
os.makedirs(os.path.dirname(DZ), exist_ok=True)
if not os.path.exists(DZ):
    !wget -q --show-progress -O "{DZ}.part" "https://zenodo.org/records/16422779/files/Dangerous%20Items.zip?download=1" && mv "{DZ}.part" "{DZ}"
!rm -rf /content/src/dangerous && mkdir -p /content/src/dangerous && unzip -q "{DZ}" -d /content/src/dangerous
!find /content/src/dangerous -maxdepth 3 -type d | head -20

<div dir="rtl">

**3b.** בודק מה יש בדאטהסט לפני האימון: אילו מחלקות, וכמה תיבות יש מכל אחת.

</div>

In [ ]:
# 3b. What is inside, before training: classes and boxes per class (the lesson of attempt 1)
import importlib, security_dataset as sd
importlib.reload(sd)
from collections import Counter
names = sd.find_names('/content/src/dangerous')
print('classes:', names)
cnt = {'train': Counter(), 'val': Counter()}
pairs = sd.yolo_pairs('/content/src/dangerous')
for img, lbl, split in pairs:
    for line in open(lbl):
        if line.split(): cnt[split][names.get(int(float(line.split()[0])), '?')] += 1
print(len(pairs), 'images with a label file')
for split in cnt: print(split, dict(cnt[split]))

In [ ]:
# 4. Build one dataset: gun, knife, fire, smoke. Stops here if a class is too thin.
importlib.reload(sd)
sd.WEAPONS_ZIP = '/content/src/weapons/weapons_images_2fps.zip'
sd.DFIRE_DIR = '/content/src/dfire/data'
!rm -rf /content/security/dataset
w = sd.Writer('/content/security/dataset')
sd.add_weapons(w)
sd.add_dfire(w, limit=8000)
base_val = list(w.files['val'])      # CCTV guns + D-Fire: what the old security.pt was validated on
sd.add_dangerous(w, '/content/src/dangerous', cctv=True)
w.yaml(); w.yaml('data_base.yaml', base_val)
print('images:', w.n)
sd.check_enough(w)

In [ ]:
# 5. Train: continue from security.pt, gently (SGD, low learning rate, first 10 layers frozen so fire / smoke
#    are not forgotten). Resumes from Drive after a disconnect.
from ultralytics import YOLO
EPOCHS = 20
NAME = 'security_v3'
last = f'{RUNS}/{NAME}/weights/last.pt'
if os.path.exists(last):
    print('resuming', last)
    YOLO(last).train(resume=True)
else:
    YOLO('security.pt').train(data='/content/security/dataset/data.yaml', epochs=EPOCHS, imgsz=800, batch=16,
                              device=0, project=RUNS, name=NAME, exist_ok=True, seed=0,
                              optimizer='SGD', lr0=0.002, lrf=0.1, warmup_epochs=1, freeze=10,
                              close_mosaic=4, scale=0.7, patience=8)
NEW = f'{RUNS}/{NAME}/weights/best.pt'

In [ ]:
# 6. Per-class numbers on the validation images: new model vs the old security.pt
from ultralytics import YOLO
NEW = f'{RUNS}/security_v3/weights/best.pt'
for title, weights in (('OLD security.pt', 'security.pt'), ('NEW security_v2', NEW)):
    for vname, yaml in (('all validation', 'data.yaml'), ('base: CCTV guns + D-Fire', 'data_base.yaml')):
        r = YOLO(weights).val(data='/content/security/dataset/' + yaml, imgsz=800, plots=False, verbose=False)
        print(f'\n{title} | {vname}')
        for k, c in enumerate(r.ap_class_index):
            print(f'  {r.names[int(c)]:6s} P={r.box.p[k]:.3f} R={r.box.r[k]:.3f} mAP50={r.box.ap50[k]:.3f}')

<div dir="rtl">

## הבדיקה האמיתית: UCF-Crime

מריצים את המצלמה החיה עם הגלאי החדש על אותם סרטונים מ-`live_eval/ucf_videos`, ושומרים ל-`live_eval/perception_fps3_sec3`. המסקנה נקבעת לפי המספרים כאן, לא לפי תא 6.

</div>

In [ ]:
# 7. UCF-Crime with the new detector (same videos and fps as the first measurement; resumable)
import shutil
FPS = 3
LE = f'{DRIVE}/live_eval'
shutil.copy(NEW, 'security_v3.pt')
os.environ['LIVE_EXTRA'] = 'security_v3.pt'
os.environ['LIVE_DEVICE'] = '0'
!python live/perceive.py --gt eval/ground_truth_uca.json --videos {LE}/ucf_videos --out {LE}/perception_fps{FPS}_sec3 --fps {FPS}

In [ ]:
# 8. Old vs new detector: diagnosis + alerts
for tag, perc in (('OLD', f'{LE}/perception_fps{FPS}'), ('NEW', f'{LE}/perception_fps{FPS}_sec3')):
    print(f'\n#################### {tag} detector')
    !python live/diagnose.py --gt eval/ground_truth_uca.json --perception {perc} --name diagnose_{tag}
    !python live/eval_alerts.py --gt eval/ground_truth_uca.json --perception {perc} --scenarios knife,weapon,snatch --name alerts_{tag}
!mkdir -p {LE}/sec3 && cp live/results/diagnose_* live/results/alerts_* {LE}/sec3/

In [ ]:
# 9. Download the new model (save it as backend/security_v3.pt) and the results
from google.colab import files
files.download(NEW)
import glob
for p in sorted(glob.glob('live/results/diagnose_*.txt') + glob.glob('live/results/alerts_*.txt')):
    files.download(p)